In [ ]:
#Mounting Google Drive to Google Collab

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Install and Importing required libraries

import torch
import torchvision
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
import torch.nn.functional as F

from sklearn.metrics import confusion_matrix
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch import nn, optim
from sklearn.metrics import f1_score
from PIL import Image


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


#ResNet Implementation

In [ ]:
#Data loading and Augmentation

data_dir = '/content/drive/MyDrive/Machine Learning/Project3_MonReader/data/images'

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(data_dir + "/training", transform=train_transforms)
test_dataset = datasets.ImageFolder(data_dir + "/testing", transform=test_transforms)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train Classes:", train_dataset.classes)
print("Test Classes:", test_dataset.classes)

In [ ]:
#Load Pre-Trained Model

resnet_model = models.resnet18(pretrained=True) #from torchvision models

# Replace final layer for binary classification
num_features = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_features, 2)

resnet_model = resnet_model.to(device)

In [ ]:
#Loss and Optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet_model.parameters(), lr=0.0001)

In [ ]:
#Training Loop

def train_model(resnet_model, train_loader, test_loader, epochs=5):
    for epoch in range(epochs):
        resnet_model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = resnet_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        test_f1 = test_model(resnet_model, test_loader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss:.2f}, Test F1: {test_f1:.4f}")

def test_model(resnet_model, loader):
    resnet_model.eval()
    preds = []
    targets = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = resnet_model(images)
            _, predicted = torch.max(outputs, 1)

            preds.extend(predicted.cpu().numpy())
            targets.extend(labels.numpy())

    return f1_score(targets, preds)

In [ ]:
#Training the model

train_model(resnet_model, train_loader, test_loader, epochs=5)

In [ ]:
#Saving the model -- in google collab

torch.save(resnet_model.state_dict(), "/content/drive/MyDrive/Machine Learning/Project3_MonReader/page_flip_model.pth")

In [ ]:
#How to Download model if required

#from google.colab import files
#files.download("/content/page_flip_model.pth")

In [ ]:
#Function to predict a single image

def predict_image(path):
    resnet_model.eval()

    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor()
    ])

    image = Image.open(path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = resnet_model(image)
        _, pred = torch.max(output, 1)

    classes = train_dataset.classes
    print("Prediction:", classes[pred.item()])

In [ ]:
#calling predict_image function with specifying the exact file path of the image

predict_image('/content/drive/MyDrive/Machine Learning/Project3_MonReader/data/images/testing/flip/0001_000000020.jpg')

In [ ]:
#calling predict_image function with specifying the exact file path of the image

predict_image('/content/drive/MyDrive/Machine Learning/Project3_MonReader/data/images/testing/notflip/0001_000000004.jpg')

In [ ]:
#Function to predict all images inside a folder -- looping until it reads all files in a folder

def predict_folder(folder_path, batch_size=32):
    resnet_model.eval()

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    # Get all image paths
    image_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    classes = train_dataset.classes
    results = []

    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        images = []

        for path in batch_paths:
            img = Image.open(path).convert("RGB")
            img = transform(img)
            images.append(img)

        images = torch.stack(images).to(device)

        with torch.no_grad():
            outputs = resnet_model(images)
            _, preds = torch.max(outputs, 1)

        for path, pred in zip(batch_paths, preds.cpu().numpy()):
            results.append((path, classes[pred]))

    return results

In [ ]:
#Specifying the locaiton of the folder
#Running predct_folder function by calling the folder path
#storing the results

folder = '/content/drive/MyDrive/Machine Learning/Project3_MonReader/data/images/testing/flip'
results_1 = predict_folder(folder)

In [ ]:
#For loop to show the prediction from all the files in the foler -- for this instance, set to 50 results for now.
#Used os.path.basename function just to specificy the main filename instead of showing the whole directory and file name

for path, label in results_1[:50]:
    filename = os.path.basename(path)
    print(f"{filename} -> {label}")

In [ ]:
#Specifying the locaiton of the folder
#Running predct_folder function by calling the folder path
#storing the results

folder = '/content/drive/MyDrive/Machine Learning/Project3_MonReader/data/images/testing/notflip'
results_2 = predict_folder(folder)

In [ ]:
#For loop to show the prediction from all the files in the foler -- for this instance, set to 50 results for now.
#Used os.path.basename function just to specificy the main filename instead of showing the whole directory and file name

for path, label in results_2[:50]:
    filename = os.path.basename(path)
    print(f"{filename} -> {label}")

#EfficientNet Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

What this does:

torch → core deep learning library

nn → neural network layers

optim → optimizer (Adam, SGD)

datasets → loads images

models → contains EfficientNet

DataLoader → creates batches

In [ ]:
#EfficientNet expects images around 224x224 (for B0 version).

train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

What this does:

Resize → ensures correct input size

Flip/Rotate → augmentation (helps generalization)

ToTensor → converts image to numbers (required)

In [ ]:
#Loading Dataset

data_dir = '/content/drive/MyDrive/Machine Learning/Project3_MonReader/data/images'

train_dataset = datasets.ImageFolder(data_dir + "/training", transform=train_transforms)
test_dataset = datasets.ImageFolder(data_dir + "/testing", transform=test_transforms)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Classes:", train_dataset.classes)

Important:

Folder names become labels automatically.

If your folders are:


```
flip/
notflip/
```

Then:


```
flip = 0
notflip = 1
```



In [ ]:
#Loading the model -- EfficientNet

en_model = models.efficientnet_b0(pretrained=True)

What happens here:

*   Loads EfficientNet architecture
*   Loads weights trained on ImageNet
*   Model already knows general image features







In [ ]:
#Modify Final Layer

num_features = en_model.classifier[1].in_features
en_model.classifier[1] = nn.Linear(num_features, 2)

ImageNet has 1000 classes.
You only have 2: flip / notflip.

EfficientNet final structure:
```
model.classifier = [Dropout, Linear(1000)]
```

We replace:
```
Linear(1000) → Linear(2)
```

Now model Outputs:
```
[score_flip, score_notflip]
```





In [ ]:
#Moving model to GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
en_model = en_model.to(device)

In [ ]:
#Loss & Optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(en_model.parameters(), lr=1e-4)

Why CrossEntropyLoss?


*   This is classification
*   Output is raw scores (logits)
*   Labels are integers (0 or 1)






In [ ]:
#Training Function

from sklearn.metrics import f1_score

def evaluate(en_model, loader):
    en_model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = en_model(images)
            _, predicted = torch.max(outputs, 1)
            preds.extend(predicted.cpu().numpy())
            targets.extend(labels.numpy())

    return f1_score(targets, preds)

def train_model_EfficientNet(en_model, train_loader, test_loader, epochs=5):
    for epoch in range(epochs):
        en_model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = en_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        f1 = evaluate(en_model, test_loader)
        print(f"Epoch {epoch+1}, Loss: {running_loss:.2f}, Test F1: {f1:.4f}")

In [ ]:
#Train

train_model_EfficientNet(en_model, train_loader, test_loader, epochs=5)

#MobileNet Implementation

In [ ]:
#Importing Libraries

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

models contains MobileNetV2

datasets + DataLoader → handle your images

f1_score → evaluate performance

In [ ]:
#Define Transforms
#MobileNet expects 224x224 RGB images, same as EfficientNet:

train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet normalization
        std=[0.229, 0.224, 0.225]
    )
])

test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
#Load Datasets

data_dir = '/content/drive/MyDrive/Machine Learning/Project3_MonReader/data/images'

train_dataset = datasets.ImageFolder(data_dir + "/training", transform=train_transforms)
test_dataset  = datasets.ImageFolder(data_dir + "/testing", transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Classes:", train_dataset.classes)

In [ ]:
#Load Pre-trained MobileNetV2

mn_model = models.mobilenet_v2(pretrained=True)

# Replace final layer for 2 classes
num_features = mn_model.classifier[1].in_features
mn_model.classifier[1] = nn.Linear(num_features, 2)

mn_model = mn_model.to(device)

MobileNetV2 has a classifier:
```
Sequential(
  Dropout,
  Linear(1280, 1000)  # ImageNet 1000 classes
)
```

*   We replace Linear(1280, 1000) → Linear(1280, 2) for flip/notflip


In [ ]:
#Loss and Optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mn_model.parameters(), lr=1e-4)

*   CrossEntropyLoss → suitable for multi-class/binary classification
*   Adam → works well for small datasets

In [ ]:
#Evaluation Function (F1 Score)

def get_f1_score(mn_model, loader):
    mn_model.eval()
    preds, labels_all = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = mn_model(images)
            _, predicted = torch.max(outputs, 1)
            preds.extend(predicted.cpu().numpy())
            labels_all.extend(labels.numpy())
    return f1_score(labels_all, preds)

In [ ]:
#Training Function

def train_model_with_MobileNet(mn_model, train_loader, test_loader, epochs=5):
    for epoch in range(epochs):
        mn_model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = mn_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # Evaluate F1 on test set at the end of this epoch
        f1 = get_f1_score(mn_model, test_loader)

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss:.2f}, Test F1: {f1:.4f}")

In [ ]:
train_model_with_MobileNet(mn_model, train_loader, test_loader, epochs=5)

#Confusion Matrix for all Models

In [ ]:
def get_predictions(model, loader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    return all_labels, all_preds

In [ ]:
def plot_confusion_matrix(model, loader, model_name):

    labels, preds = get_predictions(model, loader)
    cm = confusion_matrix(labels, preds)

    plt.figure(figsize=(6,5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['NotFlip','Flip'],
        yticklabels=['NotFlip','Flip']
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

In [ ]:
plot_confusion_matrix(resnet_model, test_loader, "ResNet")

In [ ]:
plot_confusion_matrix(en_model, test_loader, "EfficientNet")

In [ ]:
plot_confusion_matrix(mn_model, test_loader, "MobileNet")

#Custom CNN (RalphNet)

PIPELINE:<br><br>

<center>Dataset
   <center>↓<br>
<center>Transforms
   <center>↓<br>
<center>DataLoader
   <center>↓<br>
<center>Custom CNN Architecture (RalphNet)
   <center>↓<br>
<center>Loss Function
   <center>↓<br>
<center>Optimizer
   <center>↓<br>
<center>Training Loop
   <center>↓<br>
<center>Evaluation (F1 Score)
   <center>↓<br>
<center>Confusion Matrix

In [ ]:
#Definining Image Transforms

# Prepare images so they are suitable for the neural network.

# CNNs require:
# same image size
# tensor format
# normalized values

data_dir = '/content/drive/MyDrive/Machine Learning/Project3_MonReader/data/images'

train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(), #for data augmentation -- flipping the image for getting richer information
    transforms.RandomRotation(10), #for ddata augmentation -- rotating to x degrees
    transforms.ToTensor()
])

test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
#Loading the dataset

train_dataset = datasets.ImageFolder(data_dir+"/training", transform=train_transforms)
test_dataset = datasets.ImageFolder(data_dir+"/testing", transform=test_transforms)

What ImageFolder does :

*datasets.ImageFolder*

It:

*   Reads folder names as class labels
*   Loads images
*   Applies transforms





In [ ]:
#Create Dataloaders

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

What DataLoader does:<br><br>



Dataset -> Split into batches -> Feed batches to model<br><br>

Example batch:

32 images<br>
32 labels<br><br>

Why batching matters:

*   speeds up GPU training

*   stabilizes gradient updates

In [ ]:
#Define Custom CNN (RalphNet)

class RalphNet(nn.Module):

    def __init__(self):
        super(RalphNet, self).__init__()

        self.conv1 = nn.Conv2d(3,32,3,padding=1)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.conv3 = nn.Conv2d(64,128,3,padding=1)

        self.pool = nn.MaxPool2d(2,2)

        self.fc1 = nn.Linear(128*28*28,256)
        self.fc2 = nn.Linear(256,2)

    def forward(self,x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0),-1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

**Understanding Each Layer**<br><br>

nn.Conv2d -> Performs convolution filtering<br>
```
Input image -> Edge detector -> Texture detector -> Shape detector
```
Output becomes feature maps.<br><br>


---
<br>
ReLU

```
ReLU(x) = max(0,x)
```
Purpose:

*   introduces non-linearity

*   allows network to learn complex patterns
<br>


---
<br>
MaxPool2d -> Reduces spatial size.

```
224x224 -> 112x112 -> 56x56 -> 28x28
```
Benefits:

* reduces computation
* keeps strongest features
<br>

---
<br>
Flatten Layer

```
Feature maps -> 1D vector

i.e
128 x 28 x 28 -> 100352 values
```
This becomes input for fully connected layers.
<br><br>


---
<br>
Fully Connected Layers


```
Feature vector -> Dense layer -> Classifier

Output:
[score_flip, score_notflip]
```
The highest score becomes the prediction.
<br><br>





In [ ]:
#Initialize the model

ralphnet_model = RalphNet().to(device)

In [ ]:
#Define Loss Function

criterion = nn.CrossEntropyLoss()

Purpose -> Measures prediction error.<br><br>

Example:
```
Prediction: Flip
Actual: NotFlip
Loss increases
```
Training tries to minimize this loss.



In [ ]:
#Define Optimizer

optimizer = torch.optim.Adam(ralphnet_model.parameters(), lr=0.0001)

Purpose -> Updates weights to reduce loss.<br><br>

Process:


```
Forward pass -> Compute loss -> Backpropagation -> Optimizer updates weights
```



In [ ]:
#Training Loop

def train_ralphnet(ralphnet_model, train_loader, epochs):

    for epoch in range(epochs):

        ralphnet_model.train()
        running_loss = 0

        for images, labels in train_loader:

            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = ralphnet_model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        print("Epoch:",epoch+1,"Loss:",running_loss)

In [ ]:
#Definte the Evaluation function through F1 score

def get_f1_score(ralphnet_model, loader):

    ralphnet_model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = ralphnet_model(images)
            _, preds = torch.max(outputs,1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    return f1_score(all_labels, all_preds)

In [ ]:
#Training the model

train_ralphnet(ralphnet_model, train_loader, epochs=5)

In [ ]:
f1 = get_f1_score(ralphnet_model, test_loader)

print("RalphNet F1 Score:", f1)

In [ ]:
plot_confusion_matrix(ralphnet_model, test_loader, "RalphNet")

## Lion Optimizer Implementation

In [ ]:
#Installing and importing Lion Optimizer

!pip install lion-pytorch
from lion_pytorch import Lion

In [ ]:
# Initialize model AGAIN (important)
ralphnet_model_lion = RalphNet().to(device)

In [ ]:
# Lion optimizer
optimizer = Lion(ralphnet_model_lion.parameters(),
                 lr=1e-4,
                 weight_decay=1e-2
                 )

In [ ]:
# Train
train_ralphnet(ralphnet_model_lion, train_loader, epochs=5)

In [ ]:
# Evaluate
lion_f1 = get_f1_score(ralphnet_model_lion, test_loader)

print("Lion F1:", lion_f1)

### Implementing Dropout on together with Lion Optimizer

In [ ]:
#Applying Dropout to custom CNN model

class RalphNet(nn.Module):
    def __init__(self):
        super(RalphNet, self).__init__()

        # Convolution layers
        self.conv1 = nn.Conv2d(3,32,3,padding=1)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.conv3 = nn.Conv2d(64,128,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)

        # Fully connected layers
        self.fc1 = nn.Linear(128*28*28,256)
        self.dropout = nn.Dropout(p=0.5)  # 50% dropout
        self.fc2 = nn.Linear(256,2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)  # flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)         # apply dropout only during training
        x = self.fc2(x)
        return x

In [ ]:
#Reinitiliaze the model

ralphnet_model_lion_with_dropout = RalphNet().to(device)

In [ ]:
#Applying Lion Optimizer

optimizer = Lion(
    ralphnet_model_lion_with_dropout.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

In [ ]:
#Training the model
#dropout will be active during training

train_ralphnet(ralphnet_model_lion_with_dropout, train_loader, epochs=5)

In [ ]:
#Evaluating F1 score

f1 = get_f1_score(ralphnet_model_lion_with_dropout, test_loader)
print("RalphNet + Lion + Dropout F1:", f1)

####Optimizing hyperparameters for better F1 score for Lion + Dropout

In [ ]:
#Applying Dropout to custom CNN model

class RalphNet(nn.Module):
    def __init__(self):
        super(RalphNet, self).__init__()

        # Convolution layers
        self.conv1 = nn.Conv2d(3,32,3,padding=1)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.conv3 = nn.Conv2d(64,128,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)

        # Fully connected layers
        self.fc1 = nn.Linear(128*28*28,256)
        self.dropout = nn.Dropout(p=0.3)  # change from 50% to 30% dropout
        self.fc2 = nn.Linear(256,2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)  # flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)         # apply dropout only during training
        x = self.fc2(x)
        return x

In [ ]:
#Reinitiliaze the model

ralphnet_model_lion_with_dropout = RalphNet().to(device)

In [ ]:
#Applying Lion Optimizer

optimizer = Lion(
    ralphnet_model_lion_with_dropout.parameters(),
    lr=3e-4,
    weight_decay=1e-2
)

In [ ]:
#Training the model
#dropout will be active during training

train_ralphnet(ralphnet_model_lion_with_dropout, train_loader, epochs=10) #increased the epoch for convergence

In [ ]:
#Evaluating F1 score

f1 = get_f1_score(ralphnet_model_lion_with_dropout, test_loader)
print("RalphNet + Lion + Dropout F1:", f1)

####Implementing Early Stopping

What does Early Stopping do:


*   Stops training when the model stops improving
*   Prevents overfitting
*   Saves training time<br><br>

Goal:<br>
*   Monitor = F1 score on validation/test set
*   Goal = maximize F1<br><br>

Important Concept:<br>
*   best_f1 → best score seen so far
*   patience → how many epochs to wait before stopping






In [ ]:
#Training Function with earlystopping logic

def train_ralphnet_earlystop(model, train_loader, test_loader, epochs=20, patience=3):

    best_f1 = 0
    counter = 0

    for epoch in range(epochs):

        model.train()
        running_loss = 0

        for images, labels in train_loader:

            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # 🔹 Evaluate F1 after each epoch
        f1 = get_f1_score(model, test_loader)

        print(f"Epoch {epoch+1} | Loss: {running_loss:.2f} | F1: {f1:.4f}")

        # 🔹 Check improvement
        if f1 > best_f1:
            best_f1 = f1
            counter = 0

            # Save best model
            torch.save(model.state_dict(), "best_ralphnet_dropout_lion.pth")

        else:
            counter += 1

        # 🔹 Early stopping condition
        if counter >= patience:
            print("Early stopping triggered")
            break

    print("Best F1:", best_f1)

In [ ]:
#Reinitializing the model

ralphnet_model_lion_with_dropout = RalphNet().to(device)

In [ ]:
#Defining Optimizer again

optimizer = Lion(
    ralphnet_model_lion_with_dropout.parameters(),
    lr=3e-4,
    weight_decay=1e-2
)

In [ ]:
#Training the model with early stopping

train_ralphnet_earlystop(
    ralphnet_model_lion_with_dropout,
    train_loader,
    test_loader,
    epochs=20,
    patience=3
)

In [ ]:
#Loading the best model

ralphnet_model_lion_with_dropout.load_state_dict(
    torch.load("best_ralphnet_dropout_lion.pth")
)

In [ ]:
#Final F1 score

f1 = get_f1_score(ralphnet_model_lion_with_dropout, test_loader)
print("Final F1 after Early Stopping:", f1)

#EasyOCR Implementation

*PIPELINE:*

<div align="center">

Image  
↓  
EasyOCR  
↓  
Predicted Text  
↓  
Save to .txt  
↓  
Compare with Ground Truth  
↓  
Generate Character Error Rate

</div>

In [ ]:
!pip install easyocr scikit-learn

In [ ]:
import easyocr
import cv2
import os
import re
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
folder_path = '/content/drive/MyDrive/OCR_images/'

In [ ]:
image_extensions = {'.png', '.jpg', '.jpeg'}

image_files = [
    file for file in os.listdir(folder_path)
    if os.path.splitext(file)[1].lower() in image_extensions
]

print("Images found:", image_files)

In [ ]:
# Create full file paths for each image
# (This tells Python exactly where each image is located)

image_paths = [
    os.path.join(folder_path, file)
    for file in image_files
]

print(image_paths)

In [ ]:
#Initialize OCR

reader = easyocr.Reader(['en'])

In [ ]:
def normalize(text):
    text = text.lower()
    text = text.replace("-", "")
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
results_data = []

for img_path in image_paths:

    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    results = reader.readtext(img_path)

    detected_text = ""

    for (bbox, text, prob) in results:

        detected_text += text + " "

        top_left = tuple(map(int, bbox[0]))
        bottom_right = tuple(map(int, bbox[2]))

        cv2.rectangle(img_rgb, top_left, bottom_right, (0, 255, 0), 2)

        cv2.putText(
            img_rgb,
            text,
            top_left,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 0, 0),
            2
        )

    detected_text = detected_text.strip()

    filename = os.path.basename(img_path)

    results_data.append((filename, detected_text))

    # create results folder inside your folder_path
    results_dir = os.path.join(folder_path, "results")
    os.makedirs(results_dir, exist_ok=True)

    # save image with bounding boxes inside results folder
    output_img_path = os.path.join(results_dir, "boxed_" + filename)
    cv2.imwrite(output_img_path, cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))

    print("\nProcessed:", filename)
    print("OCR:", detected_text)

In [ ]:
# create results folder (if not already created)
results_dir = os.path.join(folder_path, "results")
os.makedirs(results_dir, exist_ok=True)

# save text file inside results folder
output_file = os.path.join(results_dir, "ocr_results.txt")

with open(output_file, "w") as f:
    for filename, text in results_data:
        f.write(f"{filename}: {text}\n")

print("Saved OCR text to:", output_file)

In [ ]:
for img_path in image_paths:

    img = cv2.imread(img_path)

    results_bbox = reader.readtext(img)

    for (bbox, text, _) in results_bbox:

        (top_left, top_right, bottom_right, bottom_left) = bbox

        top_left = tuple(map(int, top_left))
        bottom_right = tuple(map(int, bottom_right))

        cv2.rectangle(img, top_left, bottom_right, (0,255,0), 2)

        cv2.putText(
            img,
            text,
            top_left,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,255,0),
            2
        )

    plt.figure(figsize=(12,8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(img_path))
    plt.axis('off')
    plt.show()

In [ ]:
# Create results folder path
results_folder = os.path.join(folder_path, "results")

# Create the folder if it doesn't exist
os.makedirs(results_folder, exist_ok=True)

# Define output file inside results folder
output_file = os.path.join(results_folder, "ocr_draft_for_ground_truth.txt")

# Write OCR draft
with open(output_file, "w") as f:
    for filename, text in results_data:
        f.write(f"{filename}:\n{text}\n\n---\n\n")

print("Draft saved at:", output_file)

In [ ]:
# Ground truth (YOU must fill this manually)

ground_truth = {
    "IMG_6441.jpg": "being uncertain and not knowing, the more comfortable you will feel in knowing what you don't know. Uncertainty removes our judgments of others; it preempts the unnecessary stereotyping and biases that we otherwise feel when we see somebody on TV, in the office, Or on the street. Uncertainty also relieves us of our judgment of ourselves. We don't know if we're lovable or not; we dont know how attractive we are; we dont know how successful we could potentially become. The only way to achieve these things is to remain uncertain of them and be open to finding them out through experience. Uncertainty is the root of all progress and all growth. As the old adage goes, the man who believes he knows everything learns nothing. We cannot learn anything without first not knowing something. The more we admit we do not know, the more opportunities we gain to learn. Our values are imperfect and incomplete, and to assume that they are perfect and complete is to put us in a dangerously dogmatic mindset that breeds entitlement and avoids responsibility. The only way to solve our problems is to first admit that our actions and beliefs up to this point have been wrong and are not working. This openness to being wrong must exist for any real change or growth to take place. Before we can look at our values and prioritizations and change them into better, healthier ones, we mst first become uncertain of our current values. We must intellectually strip them away, see their faults and biases, see how they don't fit in with much of the rest of the world, to stare"
}

In [ ]:
# Function to calculate Character Error Rate (CER) between ground truth and OCR predicted text

def cer(gt, pred):

    gt = normalize(gt)
    pred = normalize(pred)

    dp = [[0] * (len(pred) + 1) for _ in range(len(gt) + 1)]

    for i in range(len(gt) + 1):
        dp[i][0] = i
    for j in range(len(pred) + 1):
        dp[0][j] = j

    for i in range(1, len(gt) + 1):
        for j in range(1, len(pred) + 1):

            if gt[i - 1] == pred[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )

    return dp[-1][-1] / max(len(gt), 1)

In [ ]:
# Calculate and display CER score for each image and the overall average CER

cers = []

for filename, pred_text in results_data:

    if filename in ground_truth:

        gt = ground_truth[filename]
        pred = pred_text

        score = cer(gt, pred)
        cers.append(score)

        print("\nFile:", filename)
        print("CER:", score)

print("\n🔥 Average CER:", sum(cers) / len(cers))

#GLM-OCR

| EasyOCR                         | GLM-OCR                                      |
| ------------------------------- | -------------------------------------------- |
| Detect + recognize text regions | End-to-end document understanding            |
| Returns bounding boxes + text   | Usually returns structured text directly     |
| Local model (PyTorch)           | Often API / model-based (depends on version) |


In [ ]:
# Install required libraries for OCR evaluation and transformer-based vision-language models

!pip install git+https://github.com/huggingface/transformers.git
!pip install transformers accelerate torch torchvision qwen-vl-utils sentencepiece| bitsandbytes
!pip install jiwer

In [ ]:
# Import transformer-based vision-language model components and image processing tools

from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
from jiwer import wer
from jiwer import cer

import torch
import os

In [ ]:
# Import login utility to authenticate with Hugging Face Hub

from huggingface_hub import login

login("<redacted>")

In [ ]:
# Load pretrained GLM-OCR vision-language model and its processor from Hugging Face

model_id = "zai-org/GLM-OCR"

processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True
)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

In [ ]:
# Run GLM-OCR model inference on a single image to extract text using a vision-language prompt

folder_path = '/content/drive/MyDrive/OCR_images/'
filename = 'IMG_6441.jpg'

img_path = os.path.join(folder_path, filename)

image = Image.open(img_path)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Extract all text from this page accurately."}
        ]
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(
    text=[text],
    images=[image],
    return_tensors="pt"
).to(model.device)

generated_ids = model.generate(
    **inputs,
    max_new_tokens=250
)

response = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

print(response)

In [ ]:
# Compare OCR output with ground truth using WER and compute approximate accuracy percentage

ground_truth = """
being uncertain and not knowing, the more comfortable you will feel in knowing what you don't know. Uncertainty removes our judgments of others; it preempts the unnecessary stereotyping and biases that we otherwise feel when we see somebody on TV, in the office, Or on the street. Uncertainty also relieves us of our judgment of ourselves. We don't know if we're lovable or not; we don't know how attractive we are; we don't know how successful we could potentially become. The only way to achieve these things is to remain uncertain of them and be open to finding them out through experience. Uncertainty is the root of all progress and all growth. As the old adage goes, the man who believes he knows everything learns nothing. We cannot learn anything without first not knowing something. The more we admit we do not know, the more opportunities we gain to learn. Our values are imperfect and incomplete, and to assume that they are perfect and complete is to put us in a dangerously dogmatic mindset that breeds entitlement and avoids responsibility. The only way to solve our problems is to first admit that our actions and beliefs up to this point have been wrong and are not working. This openness to being wrong must exist for any real change or growth to take place. Before we can look at our values and prioritizations and change them into better, healthier ones, we must first become uncertain of our current values. We must intellectually strip them away, see their faults and biases, see how they don't fit in with much of the rest of the world, to stare
"""

ocr_output = response

score = wer(ground_truth, ocr_output)

accuracy = (1 - score) * 100

print(f"WER: {score}")
print(f"Approx Accuracy: {accuracy:.2f}%")

In [ ]:
# Compare OCR output with ground truth using Character Error Rate (CER) and compute approximate accuracy percentage

score = cer(ground_truth, ocr_output)

accuracy = (1 - score) * 100

print(f"CER: {score}")
print(f"Approx Accuracy: {accuracy:.2f}%")

In [ ]:
# Print raw string representations of ground truth and OCR output to inspect hidden characters and formatting issues

print(repr(ground_truth))
print(repr(ocr_output))

#Mutimodal LLM using Qwen2-VL-2B

In [ ]:
folder_path = '/content/drive/MyDrive/OCR_images/'
filename = 'IMG_6441.jpg'

img_path = os.path.join(folder_path, filename)

image = Image.open(img_path)

In [ ]:
# Enable synchronous CUDA error reporting to help debug GPU-related issues
# CUDA_LAUNCH_BLOCKING should be set first before importing torch

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
# Import Qwen2-VL vision-language model, processor, and required image/torch utilities

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image
import torch

In [ ]:
#Loading Qwen2VL Model

qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
#Load Image

image = Image.open(img_path)

# FORCE RGB (critical fix)
image = image.convert("RGB")

# resize to avoid GPU issues
#image = image.resize((image.width // 2, image.height // 2))

# Resize safely for GPU
max_size = 2048
image.thumbnail((max_size, max_size))

print(image.mode, image.size)

In [ ]:
# Define structured prompt for OCR model: extract text accurately and provide a simple explanation

prompt = """
You are an OCR and text understanding system.

Your task has TWO parts:

1. OCR_TEXT:
Extract ALL visible text from the image exactly as it appears.
- Preserve line breaks
- Fix obvious hyphenation caused by line wrapping
- Do NOT summarize or change meaning

2. EXPLANATION:
Explain what the extracted text is about in simple terms.

STRICT RULES:
- Do NOT add any extra sections
- Do NOT include confidence scores
- Do NOT add commentary outside the two sections below
- Do NOT hallucinate missing text

Return ONLY in this format:

OCR_TEXT:
<extracted text here>

EXPLANATION:
<explanation of the text>
"""

In [ ]:
# Format image-text message and preprocess inputs for Qwen2-VL model inference

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }
]


# Build prompt
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Prepare inputs
inputs = processor(
    text=[text],
    images=[image],
    return_tensors="pt"
)

In [ ]:
# Move tensors to correct device
inputs = {k: v.to(qwen_model.device) for k, v in inputs.items()}

In [ ]:
# Generate OCR and explanation output from Qwen2-VL model without gradient computation

with torch.no_grad():
    generated_ids = qwen_model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id
    )

In [ ]:
# Decode model-generated token IDs into readable text and print the final OCR result

response = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

print(response)

#Text Extraction Clean Up using Qwen-2.5-3B

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re

In [ ]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

clean_tokenizer = AutoTokenizer.from_pretrained(model_name)

clean_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
def extract_ocr_text(response):

    # Keep ONLY the final assistant response
    if "assistant" in response:
        response = response.rsplit("assistant", 1)[-1]

    # Extract OCR_TEXT block
    match = re.search(
        r"OCR_TEXT:\s*(.*?)\s*EXPLANATION:",
        response,
        re.DOTALL
    )

    if match:
        return match.group(1).strip()

    return response.strip()

In [ ]:
def build_cleanup_prompt(text):
    return f"""
You are a text reconstruction system.

Your task is to clean OCR-extracted text into a well-structured readable format.

STRICT RULES:
- Do NOT summarize
- Do NOT add new information
- Do NOT change meaning
- Fix broken sentences caused by line wrapping
- Merge fragmented lines into full sentences
- Reconstruct proper paragraphs
- Remove OCR artifacts
- Keep punctuation natural for reading

INPUT TEXT:
{text}

OUTPUT:
Return ONLY the cleaned text.
"""

In [ ]:
def clean_text_with_llm(raw_text):

    prompt = f"""
You are a text reconstruction system.

Your task is to clean OCR-extracted text into readable book-quality text.

RULES:
- Do NOT summarize
- Do NOT add new information
- Preserve meaning exactly
- Fix broken line wraps
- Reconstruct paragraphs
- Remove OCR artifacts
- Keep natural punctuation
- Return ONLY cleaned text

TEXT:
{raw_text}
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    input_text = clean_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = clean_tokenizer(
        input_text,
        return_tensors="pt"
    ).to(clean_model.device)

    with torch.no_grad():

        output_ids = clean_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False
        )

    response = clean_tokenizer.batch_decode(
        output_ids,
        skip_special_tokens=True
    )[0]

    # isolate assistant output
    if "assistant" in response:
        response = response.split("assistant")[-1]

    return response.strip()

In [ ]:
# Step 1: Qwen2-VL output (your existing code)
response = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

print("RAW OUTPUT:\n", response)

# Step 2: Extract OCR text only
ocr_text = extract_ocr_text(response)

print("\n--- OCR TEXT ---\n", ocr_text)

# Step 3: Clean with Qwen2.5-3B
final_clean_text = clean_text_with_llm(ocr_text)

print("\n--- CLEANED TEXT ---\n", final_clean_text)

#Sesame CSM Implementation (for conversion of text to audio)

In [ ]:
!pip install transformers accelerate soundfile sentencepiece
!pip install torch torchaudio
!pip install pydub

In [ ]:
import torch
import re
import soundfile as sf
import torchaudio

from transformers import AutoProcessor, AutoModel, CsmForConditionalGeneration, AutoModelForCausalLM
from pydub import AudioSegment
from IPython.display import Audio

In [ ]:
model_id = "sesame/csm-1b"

processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True
)

tts_model = CsmForConditionalGeneration.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

tts_model.codec_model = tts_model.codec_model.to("cuda")

tts_model.eval()

In [ ]:
print(type(tts_model))

In [ ]:
print(final_clean_text)

In [ ]:
def split_text(text, max_chars=250):

    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current = ""

    for sentence in sentences:

        if len(current) + len(sentence) < max_chars:
            current += " " + sentence
        else:
            chunks.append(current.strip())
            current = sentence

    if current:
        chunks.append(current.strip())

    return chunks

In [ ]:
print(type(tts_model))

print("hf_device_map:")
print(getattr(tts_model, "hf_device_map", None))

for name, module in tts_model.named_children():
    try:
        dev = next(module.parameters()).device
    except:
        dev = "no params"

    print(name, dev)

In [ ]:
chunks = split_text(final_clean_text)

audio_files = []

for i, chunk in enumerate(chunks):

    inputs = processor(
        chunk,
        return_tensors="pt"
    ).to(tts_model.device)

    with torch.no_grad():
        tokens = tts_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=True
        )

    if isinstance(tokens, (list, tuple)):
        tokens = tokens[0]

    # =========================
    # 🔥 MEMORY FIX SECTION
    # =========================
    torch.cuda.empty_cache()

    tokens = tokens.detach()

    decoded = tts_model.codec_model.decode(tokens)

    if hasattr(decoded, "audio_values"):
        waveform = decoded.audio_values
    elif hasattr(decoded, "waveform"):
        waveform = decoded.waveform
    else:
        waveform = decoded[0] if isinstance(decoded, (tuple, list)) else decoded

    waveform = waveform.squeeze().detach().cpu().float().numpy()

    filename = f"chunk_{i}.wav"

    import soundfile as sf
    sf.write(filename, waveform, 24000)

    # cleanup
    del inputs
    del tokens
    del decoded
    del waveform

    import gc
    gc.collect()
    torch.cuda.empty_cache()

    audio_files.append(filename)

    print("Saved:", filename)

In [ ]:
combined = AudioSegment.empty()

for file in audio_files:

    segment = AudioSegment.from_wav(file)

    combined += segment

combined.export(
    "final_audiobook.wav",
    format="wav"
)

print("Done!")

In [ ]:
Audio("final_audiobook.wav")